In [1]:
%run setup.py

Setup done ✔
Current dir: C:\projects\data-Inspector


In [11]:
from src.utils.ai_helper import (
    generate_quality_report,
    generate_debug_report
)

from src.utils.metrics_loader import load_metrics

In [12]:
data, metrics = load_metrics()

print("Metrics loaded successfully.")

Loading data...
Computing metrics...
Metrics loaded successfully.


In [13]:
metrics["coverage"]

{'% moyen de couverture des étudiants': np.float64(0.87),
 '% étudiants avec couverture ≥ 80%': 1.0,
 '% étudiants avec couverture < 50%': 0.0,
 '% moyen de densité de couverture par matière': np.float64(0.87),
 '% moyen de couverture des matières': np.float64(0.87),
 '% moyen de densité de couverture par trimestre': np.float64(0.87),
 '% couverture globale': 0.87,
 '% couverture (étudiant, matière)': 1.0,
 '% couverture (étudiant, trimestre)': 1.0}

In [14]:
metrics["noise"]

{'% notes NULL': 0.1003,
 '% notes invalides': 0.0297,
 '% note_comportement NULL': np.float64(0.0518),
 '% lignes inutilisables': 0.0,
 '% notes égales à 0': np.float64(0.0149),
 '% notes égales à 20': np.float64(0.0),
 '% notes réalistes (8-16)': np.float64(0.7528),
 'moyenne générale': np.float64(10.29),
 'médiane': np.float64(10.26),
 'écart-type': np.float64(3.11),
 '% valeurs aberrantes': 0.0171}

In [15]:
metrics["duplicates"]

{'% duplication PK étudiants': np.float64(0.0),
 '% duplication PK enseignement': np.float64(0.0),
 '% duplication PK évaluations': np.float64(0.0),
 '% duplication métier enseignement': np.float64(0.32),
 '% duplication métier évaluations': np.float64(0.0),
 '% duplication étudiant (nom + user)': np.float64(0.0),
 '% duplication user étudiant': np.float64(0.0)}

In [16]:
metrics["integrity"]

{'% notes avec activité inexistante': 0.0,
 '% activités avec enseignant inexistant': 0.0,
 '% évaluations avec étudiant inexistant': 0.0,
 '% évaluations avec matière inexistante': 0.0,
 '% enseignements avec enseignant inexistant': 0.0,
 '% enseignements avec matière inexistante': 0.0,
 '% incohérence activité ↔ enseignant/matière': 0.0,
 '% incohérence classe ↔ niveau': np.float64(0.0),
 '% incohérence trimestre ↔ année': 0.0,
 '% étudiants sans inscription': 0.0,
 '% enseignants sans activité': 0.0,
 '% matières sans activité': 0.0}

In [17]:
report = generate_quality_report(metrics, data)

print(report)

## Data Quality Engineering Report: Academic Information System

### Executive Summary
The dataset reflects a system with high **structural rigidity** but underlying **operational friction**. While referential integrity is nearly perfect—suggesting a robust relational database schema—the actual data entry process is struggling to keep pace with the academic calendar. The presence of "ghost" teaching assignments and a centralized naming convention indicates the system might be heavily reliant on automated scripts or templates that don't always align with real-world classroom dynamics.

---

### Data Structure Observations
The schema follows a traditional academic hierarchy, yet the `Activite` table reveals a "template-driven" approach. Every subject appears to follow an identical assessment pattern (Oral, Written, Exam) with fixed coefficients. 
- **Inflexibility:** This structure suggests the system cannot easily accommodate subjects with different pedagogical requirements (e.g., lab-b

In [18]:
with open("quality_report.md", "w", encoding="utf-8") as f:
    f.write(report)

print("Report saved.")

Report saved.


In [19]:
code = """
df = pd.read_csv("etudiants.csv")

print(df["id-etudiant"])
"""

error = """
KeyError: 'id-etudiant'
"""

debug_report = generate_debug_report(
    error_message=error,
    code_snippet=code
)

print(debug_report)

As a Senior Data Engineer, here is my assessment of the issue.

### 1. Explanation of the Error
A `KeyError` in pandas indicates that the label you are trying to access (`'id-etudiant'`) does not exist in the DataFrame's column index. Pandas cannot find a column that matches that string exactly.

### 2. Root Cause
The most common causes for this in a data pipeline are:
*   **Leading/Trailing Whitespace:** The CSV header might have hidden spaces (e.g., `" id-etudiant"`).
*   **Delimiter Mismatch:** If the file uses a semicolon (`;`) instead of a comma, pandas reads the entire row as a single column.
*   **Encoding Issues:** A Byte Order Mark (BOM) at the start of the file can corrupt the first column name.
*   **Case Sensitivity:** The column might be `"Id-Etudiant"` or `"ID-ETUDIANT"`.

### 3. Corrected Version
Before accessing the column, you should inspect and sanitize the column names.

```python
import pandas as pd

# 1. Load data - using sep=None with engine='python' can auto-dete